In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from src.spark_session import get_spark
from src.config import CUSTOMERS_SILVER_PATH, ORDERS_SILVER_PATH, ORDER_ITEMS_SILVER_PATH, PAYMENTS_SILVER_PATH, PRODUCTS_SILVER_PATH, ORDERS_ENRICHED_PATH, ORDER_ITEMS_WITH_PRODUCTS_PATH

In [2]:
spark = get_spark("SalesAnalysis")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/07 11:16:22 WARN Utils: Your hostname, Branimirs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.199 instead (on interface en0)
26/08/07 11:16:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/branimiranastasov/PycharmProjects/azure_retail_lakehouse/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/07 11:16:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applic

In [3]:
orders_cleaned_df = spark.read.option("header", True).parquet(str(ORDERS_SILVER_PATH))
orders_cleaned_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)



In [4]:
customers_cleaned_df = spark.read.option("header", True).parquet(str(CUSTOMERS_SILVER_PATH))
customers_cleaned_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [5]:
orders_without_customer = (
    orders_cleaned_df
    .join(customers_cleaned_df, on="customer_id", how="left_anti")
)

print("Orders without customers: ", orders_without_customer.count())


Orders without customers:  0


In [6]:
customers_without_orders = (
    customers_cleaned_df
    .join(orders_cleaned_df, on="customer_id", how="left_anti")
)

print("Customers without orders: ", customers_without_orders.count())

Customers without orders:  0


In [7]:
orders_enriched_df = (
    orders_cleaned_df
    .join(customers_cleaned_df, on="customer_id", how="left")
)

orders_enriched_df.show(10, truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |order_id                        |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|delivery_delay_days|is_late|customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+--------------------------------+-------

In [8]:
orders_before_join = orders_cleaned_df.count()
orders_after_join = orders_enriched_df.count()

print("Before join: ", orders_before_join)
print("After join: ", orders_after_join)
print("Rows count unchanged: ", orders_before_join == orders_after_join)

Before join:  99441
After join:  99441
Rows count unchanged:  True


In [9]:
distinct_orders_after_join = (
    orders_enriched_df
    .select("order_id")
    .distinct()
    .count()
)

print("Distinct order IDs after join: ", distinct_orders_after_join)

Distinct order IDs after join:  99441


In [10]:
orders_by_state = (
    orders_enriched_df
    .groupBy("customer_state")
    .agg(
        F.count("*").alias("orders_count"),
    )
    .orderBy(F.col("orders_count").desc())
)

orders_by_state.show(30)

+--------------+------------+
|customer_state|orders_count|
+--------------+------------+
|            sp|       41746|
|            rj|       12852|
|            mg|       11635|
|            rs|        5466|
|            pr|        5045|
|            sc|        3637|
|            ba|        3380|
|            df|        2140|
|            es|        2033|
|            go|        2020|
|            pe|        1652|
|            ce|        1336|
|            pa|         975|
|            mt|         907|
|            ma|         747|
|            ms|         715|
|            pb|         536|
|            pi|         495|
|            rn|         485|
|            al|         413|
|            se|         350|
|            to|         280|
|            ro|         253|
|            am|         148|
|            ac|          81|
|            ap|          68|
|            rr|          46|
+--------------+------------+



In [11]:
late_delivery_by_state = (
    orders_enriched_df
    .filter(F.col("order_status") == "delivered")
    .groupBy("customer_state")
    .agg(
        F.count("*").alias("delivered_orders"),
        F.sum(F.col("is_late").cast("int")).alias("late_orders"),
    )
    .withColumn(
        "later_percentage",
        F.round(F.col("late_orders") / F.col("delivered_orders") * 100, 2)
    )
    .orderBy(F.col("later_percentage").desc())
)

late_delivery_by_state.show(30)

+--------------+----------------+-----------+----------------+
|customer_state|delivered_orders|late_orders|later_percentage|
+--------------+----------------+-----------+----------------+
|            al|             397|         85|           21.41|
|            ma|             717|        125|           17.43|
|            se|             335|         51|           15.22|
|            pi|             476|         66|           13.87|
|            ce|            1279|        176|           13.76|
|            rr|              41|          5|            12.2|
|            ba|            3256|        396|           12.16|
|            rj|           12350|       1495|           12.11|
|            pa|             946|        106|           11.21|
|            es|            1995|        214|           10.73|
|            pb|             517|         54|           10.44|
|            to|             274|         27|            9.85|
|            ms|             701|         68|          

In [12]:
orders_enriched_df.coalesce(2).write.mode("overwrite").parquet(str(ORDERS_ENRICHED_PATH))

In [13]:
saved_orders_enriched_df = spark.read.option("header", True).parquet(str(ORDERS_ENRICHED_PATH))
saved_orders_enriched_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [14]:
order_items_clean_df = spark.read.option("header", True).parquet(str(ORDER_ITEMS_SILVER_PATH))
order_items_clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- item_total: decimal(13,2) (nullable = true)



In [15]:
order_items_enriched_df = order_items_clean_df.join(
    orders_enriched_df.select(
        "order_id",
        "customer_id",
        "customer_unique_id",
        "customer_state",
        "order_status",
        "order_purchase_timestamp",
        "delivery_days",
        "delivery_delay_days",
        "is_late"
    ),
    on="order_id",
    how="left",
)

order_items_enriched_df.show(5, truncate=False)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+----------+--------------------------------+--------------------------------+--------------+------------+------------------------+-------------+-------------------+-------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price |freight_value|item_total|customer_id                     |customer_unique_id              |customer_state|order_status|order_purchase_timestamp|delivery_days|delivery_delay_days|is_late|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+----------+--------------------------------+--------------------------------+--------------+------------+------------------------+-------------+-------------------+-------+
|00018f77f2f0320

In [16]:
order_items_enriched_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- item_total: decimal(13,2) (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- delivery_days: integer (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)
 |-- is_late: boolean (nullable = true)



In [17]:
items_before = order_items_clean_df.count()
items_after = order_items_enriched_df.count()

print("Before join: ", items_before)
print("After join: ", items_after)
print("Row count preserved: ", items_before == items_after)

Before join:  112650
After join:  112650
Row count preserved:  True


In [18]:
order_revenue_df = (
    order_items_enriched_df
    .groupBy("order_id")
    .agg(
        F.sum("price").alias("product_revenue"),
        F.sum("freight_value").alias("freight_revenue"),
        F.sum("item_total").alias("total_order"),
        F.count("*").alias("item_rows")
    )
)

order_revenue_df.orderBy(F.col("total_order").desc()).show(10, truncate=False)

+--------------------------------+---------------+---------------+-----------+---------+
|order_id                        |product_revenue|freight_revenue|total_order|item_rows|
+--------------------------------+---------------+---------------+-----------+---------+
|03caa2c082116e1d31e67e9ae3700499|13440.00       |224.08         |13664.08   |8        |
|736e1922ae60d0d6a89247b851902527|7160.00        |114.88         |7274.88    |4        |
|0812eb902a67711a1cb742b3cdaa65ae|6735.00        |194.31         |6929.31    |1        |
|fefacc66af859508bf1a7934eab1e97f|6729.00        |193.21         |6922.21    |1        |
|f5136e38d1a14a4dbd87dff67da82701|6499.00        |227.66         |6726.66    |1        |
|2cc9089445046817a7539d90805e6e5a|5934.60        |146.94         |6081.54    |6        |
|a96610ab360d42a2e5335a3998b4718a|4799.00        |151.34         |4950.34    |1        |
|b4c4b76c642808cbe472a32b86cddc95|4599.90        |209.54         |4809.44    |2        |
|199af31afc78c699f0db

In [19]:
products_clean_df = spark.read.option("header", True).parquet(str(PRODUCTS_SILVER_PATH))
products_clean_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



In [20]:
items_without_product = (
    order_items_clean_df
    .join(products_clean_df.select("product_id"), on="product_id", how="left_anti")
)

print("Order items without matching product id: ", items_without_product.count())

Order items without matching product id:  0


In [21]:
order_items_with_products_df = (
    order_items_enriched_df
    .join(
        products_clean_df.select(
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ),
        on="product_id",
        how="left",
    )
)

order_items_with_products_df.show(5, truncate=False)

+--------------------------------+--------------------------------+-------------+--------------------------------+-------------------+------+-------------+----------+--------------------------------+--------------------------------+--------------+------------+------------------------+-------------+-------------------+-------+---------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |order_id                        |order_item_id|seller_id                       |shipping_limit_date|price |freight_value|item_total|customer_id                     |customer_unique_id              |customer_state|order_status|order_purchase_timestamp|delivery_days|delivery_delay_days|is_late|product_category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+--------------------------------+-------------+--------------------------------+-------------------+------+-------------+-----

In [42]:
order_items_with_products_df.coalesce(2).write.mode("overwrite").parquet(str(ORDER_ITEMS_WITH_PRODUCTS_PATH))

In [22]:
rows_before = order_items_enriched_df.count()
rows_after = order_items_with_products_df.count()

print("Before: ", rows_before)
print("After: ", rows_after)
print("Row count preserved: ", rows_before == rows_after)

Before:  112650
After:  112650
Row count preserved:  True


In [23]:
distinct_items_keys = (
    order_items_with_products_df
    .select("order_id", "order_item_id")
    .distinct()
    .count()
)

print("One row per order item: ", distinct_items_keys == rows_after)

One row per order item:  True


In [24]:
category_revenue_df = (
    order_items_with_products_df
    .groupBy("product_category_name")
    .agg(
        F.sum("price").alias("product_revenue"),
        F.sum("freight_value").alias("freight_revenue"),
        F.sum("item_total").alias("total_revenue"),
        F.count("*").alias("item_rows"),
        F.countDistinct("order_id").alias("order_count"),
    )
    .orderBy(F.col("total_revenue").desc())
)

category_revenue_df.show(20, truncate=False)

+----------------------+---------------+---------------+-------------+---------+-----------+
|product_category_name |product_revenue|freight_revenue|total_revenue|item_rows|order_count|
+----------------------+---------------+---------------+-------------+---------+-----------+
|beleza_saude          |1258681.34     |182566.73      |1441248.07   |9670     |8836       |
|relogios_presentes    |1205005.68     |100535.93      |1305541.61   |5991     |5624       |
|cama_mesa_banho       |1036988.68     |204693.04      |1241681.72   |11115    |9417       |
|esporte_lazer         |988048.97      |168607.51      |1156656.48   |8641     |7720       |
|informatica_acessorios|911954.32      |147318.08      |1059272.40   |7827     |6689       |
|moveis_decoracao      |729762.49      |172749.30      |902511.79    |8334     |6449       |
|utilidades_domesticas |632248.66      |146149.11      |778397.77    |6964     |5884       |
|cool_stuff            |635290.85      |84039.10       |719329.95    |

In [25]:
avg_price_per_category = (
    order_items_with_products_df
    .groupBy("product_category_name")
    .agg(F.round(F.avg("price"), 2).alias("avg_item_price"))
    .orderBy(F.col("avg_item_price").desc())
)

avg_price_per_category.show(20, truncate=False)

+----------------------------------------------+--------------+
|product_category_name                         |avg_item_price|
+----------------------------------------------+--------------+
|pcs                                           |1098.34       |
|portateis_casa_forno_e_cafe                   |624.29        |
|eletrodomesticos_2                            |476.12        |
|agro_industria_e_comercio                     |342.12        |
|instrumentos_musicais                         |281.62        |
|eletroportateis                               |280.78        |
|portateis_cozinha_e_preparadores_de_alimentos |264.57        |
|telefonia_fixa                                |225.69        |
|construcao_ferramentas_seguranca              |208.99        |
|relogios_presentes                            |201.14        |
|climatizacao                                  |185.27        |
|moveis_quarto                                 |183.75        |
|pc_gamer                               

In [26]:
payments_clean_df = spark.read.option("header", True).parquet(str(PAYMENTS_SILVER_PATH))
payments_clean_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: decimal(10,2) (nullable = true)



In [27]:
payment_type_summary = (
    payments_clean_df
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("payment_rows"),
        F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
    )
    .orderBy(F.col("total_payment_value").desc())
)

payment_type_summary.show(20, truncate=False)

+------------+------------+-------------------+
|payment_type|payment_rows|total_payment_value|
+------------+------------+-------------------+
|credit_card |76795       |12542084.19        |
|boleto      |19784       |2869361.27         |
|voucher     |5775        |379436.87          |
|debit_card  |1529        |217989.79          |
|not_defined |3           |0.00               |
+------------+------------+-------------------+



In [28]:
negative_payments = payments_clean_df.filter(F.col("payment_value") < 0)

print("Negative payment values: ", negative_payments.count())

Negative payment values:  0


In [29]:
invalid_instalments = payments_clean_df.filter(F.col("payment_installments") <= 0)

print("Non-positive installments: ", invalid_instalments.count())

Non-positive installments:  2


In [30]:
invalid_instalments.show(20, truncate=False)

+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|744bade1fcf9ff3f31d860ace076d422|2                 |credit_card |0                   |58.69        |
|1a57108394169c0b47d8f876acc9ba2d|2                 |credit_card |0                   |129.94       |
+--------------------------------+------------------+------------+--------------------+-------------+



In [31]:
payments_without_order = (
    payments_clean_df
    .join(order_items_enriched_df, on="order_id", how="left_anti")
)

print("Payments without matching orders: ", payments_without_order.count())

Payments without matching orders:  830


In [32]:
payments_without_order.show(20, truncate=False)

+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|016706d732e79b49035275c048edb2c4|1                 |credit_card |1                   |65.00        |
|01cb6d702e5233235f4125309d184bf4|1                 |boleto      |1                   |94.21        |
|03d1edbd314ca7682ec0f3e67d3763e2|1                 |credit_card |5                   |74.03        |
|08662c478bed0444d0515925af759547|1                 |credit_card |5                   |102.92       |
|16b8c07980cc1e47a31ebb35d9222807|1                 |credit_card |1                   |580.07       |
|1982a20c2a9d116da964e488d59eebe7|1                 |credit_card |3                   |34.37        |
|21a00b08cbeb5716bbb66105e3dbd850|1                 |voucher     |1               

In [33]:
payments_by_order_df = (
    payments_clean_df
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("total_payment_value"),
        F.count("*").alias("payment_row_count"),
        F.countDistinct("payment_type").alias("payment_type_count"),
        F.max("payment_installments").alias("max_installments"),
    )
)

payments_by_order_df.show(20, truncate=False)

+--------------------------------+-------------------+-----------------+------------------+----------------+
|order_id                        |total_payment_value|payment_row_count|payment_type_count|max_installments|
+--------------------------------+-------------------+-----------------+------------------+----------------+
|1970c6df1f91ef915179e223145ccd45|194.37             |1                |1                 |2               |
|eec4cdb15e74c35073cc163bfd9fa2b2|38.10              |1                |1                 |1               |
|3cabf3e464629648b819dd1bb6899bad|176.78             |1                |1                 |1               |
|5e57ff5e1c008db89fac24b76655dbe1|174.22             |1                |1                 |2               |
|c830cc216aadfbfc979193cf372cae2d|75.50              |1                |1                 |1               |
|30e934394c047a409bb861a9ecbcff43|388.36             |2                |1                 |4               |
|bf11d6d45abb67220d

In [34]:
order_value_comparison_df = (
    order_revenue_df
    .join(payments_by_order_df, on="order_id", how="inner")
    .withColumn(
        "payment_difference",
        F.round(F.col("total_payment_value") - F.col("total_order"), 2),
    )
)

order_value_comparison_df.show(20, truncate=False)

+--------------------------------+---------------+---------------+-----------+---------+-------------------+-----------------+------------------+----------------+------------------+
|order_id                        |product_revenue|freight_revenue|total_order|item_rows|total_payment_value|payment_row_count|payment_type_count|max_installments|payment_difference|
+--------------------------------+---------------+---------------+-----------+---------+-------------------+-----------------+------------------+----------------+------------------+
|00018f77f2f0320c557190d7a144bdd3|239.90         |19.93          |259.83     |1        |259.83             |1                |1                 |3               |0.00              |
|00042b26cf59d7ce69dfabb4e55b4fd9|199.90         |18.14          |218.04     |1        |218.04             |1                |1                 |3               |0.00              |
|00054e8431b9d7675808bcb819fb4a32|19.90          |11.85          |31.75      |1        |31

In [35]:
order_value_comparison_df.orderBy(F.abs(F.col("payment_difference")).desc()).show(20, truncate=False)

+--------------------------------+---------------+---------------+-----------+---------+-------------------+-----------------+------------------+----------------+------------------+
|order_id                        |product_revenue|freight_revenue|total_order|item_rows|total_payment_value|payment_row_count|payment_type_count|max_installments|payment_difference|
+--------------------------------+---------------+---------------+-----------+---------+-------------------+-----------------+------------------+----------------+------------------+
|ce6d150fb29ada17d2082f4847107665|1299.00        |104.66         |1403.66    |1        |1586.47            |1                |1                 |10              |182.81            |
|6e5fe7366a2e1bfbf3257dba0af1267f|179.19         |108.72         |287.91     |6        |406.92             |1                |1                 |10              |119.01            |
|70b742795bc441e94a44a084b6d9ce7a|269.99         |196.94         |466.93     |1        |57

In [36]:
matching_orders = order_value_comparison_df.filter(F.abs(F.col("payment_difference")) <= 0.01)
print("Orders with matching totals: ", matching_orders.count())

Orders with matching totals:  98362


In [37]:
different_orders = order_value_comparison_df.filter(F.abs(F.col("payment_difference")) > 0.01)
print("Different orders: ", different_orders.count())

Different orders:  303


In [38]:
customer_order_window = Window.partitionBy("customer_unique_id").orderBy(F.col("order_purchase_timestamp").desc())

latest_order_per_customer_id = (
    orders_enriched_df
    .withColumn(
        "row_number",
        F.row_number().over(customer_order_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

latest_order_per_customer_id.show(truncate=False)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+--------------------------------+------------------------+-----------------+--------------+
|customer_id                     |order_id                        |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|delivery_delay_days|is_late|customer_unique_id              |customer_zip_code_prefix|customer_city    |customer_state|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+-------------------+-------+--------------------------------+---------------

In [39]:
product_revenue_df = (
    order_items_with_products_df
    .groupBy("product_category_name", "product_id")
    .agg(F.round(F.sum("price"), 2).alias("product_revenue"))
)

product_revenue_df.show(5, truncate=False)

+---------------------+--------------------------------+---------------+
|product_category_name|product_id                      |product_revenue|
+---------------------+--------------------------------+---------------+
|utilidades_domesticas|ac7e981115ad47f0e051f1b8b97e73b1|252.52         |
|papelaria            |06f0e85c7892d5df893f332706340af1|5120.00        |
|papelaria            |9e18e70b32c00d98cc4cb4314dfef5dd|89.00          |
|utilidades_domesticas|d00f6c52d2730de05fb409f09ab6e732|495.07         |
|cool_stuff           |20090bcd0d43eb49feb63d53d387780b|968.00         |
+---------------------+--------------------------------+---------------+
only showing top 5 rows


In [40]:
category_revenue_window = (
    Window
    .partitionBy("product_category_name")
    .orderBy(F.col("product_revenue").desc())
)

ranked_products_df = (
    product_revenue_df
    .withColumn(
        "revenue_rank",
        F.dense_rank().over(category_revenue_window)
    )
)

ranked_products_df.show(5, truncate=False)

+-------------------------+--------------------------------+---------------+------------+
|product_category_name    |product_id                      |product_revenue|revenue_rank|
+-------------------------+--------------------------------+---------------+------------+
|agro_industria_e_comercio|11250b0d4b709fee92441c5f34122aed|9111.00        |1           |
|agro_industria_e_comercio|423a6644f0aa529e8828ff1f91003690|8043.00        |2           |
|agro_industria_e_comercio|672e757f331900b9deea127a2a7b79fd|6885.00        |3           |
|agro_industria_e_comercio|c183fd5d2abf05873fa6e1014ed9e06c|5934.60        |4           |
|agro_industria_e_comercio|2b69866f22de8dad69c976771daba91c|2990.00        |5           |
+-------------------------+--------------------------------+---------------+------------+
only showing top 5 rows


In [41]:
customers_with_orders_df = (
    customers_cleaned_df
    .join(orders_cleaned_df.select("customer_id"), on="customer_id", how="left_semi")
)

customers_with_orders_df.show(5, truncate=False)

+--------------------------------+--------------------------------+------------------------+----------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city   |customer_state|
+--------------------------------+--------------------------------+------------------------+----------------+--------------+
|000bf8121c3412d3057d32371c5d3395|1bc9b2dad6aefbfbc011508e34c8adfc|12335                   |jacarei         |sp            |
|00114026c1b7b52ab1773f317ef4880b|f4dc0a81a11d3d270ccf5a9c4b5b187b|22470                   |rio de janeiro  |rj            |
|0015f7887e2fde13ddaa7b8e385af919|866c923cde750dfc8cfbcf9d5ced0ee4|25903                   |mage            |rj            |
|001f6f1a5e902ad14e1f709a7215de11|c6b7dcd3718d1ad87f069d32a8566ce2|12460                   |campos do jordao|sp            |
|002348c1099e3229276c8ad7d4ddc702|934c19eeef04da89928f995df85cf3f8|13295                   |itupeva         |sp            |
